# BuffettRAG: LoRA fine-tune Qwen2.5-7B-Instruct (FT-r4) on ft_v7, prompt v3, Colab + Drive
Rows carry the exact serving chat (system + user as llama_provider.split_prompt builds them for PROMPT_VERSION=v3, Berkshire-only corpus); loss on the assistant answer only. LoRA rank/alpha/target modules/lr/effective batch are FT-r1's and FT-r3's (r16, alpha 32, lr 1e-4 linear, warmup 5, batch 1 x accum 2); only the schedule bookkeeping changed (steps instead of epochs). One `RUN_NAME` drives every path and name.

**Run procedure**
1. Runtime > Change runtime type > GPU (T4, L4 or A100). Keep the tab open: Colab does not run in the background on AI Pro.
2. Put `colab-bundle-7b-<DATA_NAME>.zip` (built by `scripts/ft/colab_bundle.py`) in `MyDrive/buffettrag/`. Optional dev slice: `MyDrive/buffettrag/slice_prompts.jsonl` (rows `id`, `messages`).
3. First time only: the pins for peft, bitsandbytes, unsloth_zoo, the base-model revision hash stay `None` in section a; the install cell refuses to run until they are filled (set `REQUIRE_ALL_PINS = False` for a G1 smoke run: it writes the resolved values to `pins_resolved.json` on Drive, paste them into section a).
4. Run all cells. Everything durable (full checkpoints every `SAVE_STEPS`, adapter-only copies in `adapters/step-N`, `run_meta.json`, `metrics.json`, GGUFs, `SHA256SUMS`, `slice_answers_<step>.jsonl`) goes to `MyDrive/buffettrag/run-7b-<RUN_NAME>/`.
5. Legs: set `STOP_AT_STEP` (e.g. 130 for the pilot gate) to end a leg cleanly (saves, evals, runs the slice cell, skips the export). Remove it for the next leg; training continues from the checkpoint.
6. **After a disconnect or a closed tab:** reconnect, set the same settings, Run all. The notebook remounts Drive, skips a half-written checkpoint, resumes from the newest COMPLETE one (at most `SAVE_STEPS` steps = about 33 min of work lost on a T4) and refuses to resume if the data hash or run name differ. If only the GGUF export was interrupted, set `ALLOW_EXISTING_EXPORT = True` to finish it (existing complete GGUFs are kept).
7. Take `SHA256SUMS` and the GGUFs back to the Mac; acceptance gate there: export-chain `tokpar.py` and `meta.py` on each GGUF, then the paired measurement (Q8_0 against Q4_K_M on a small subset before reading the headline).

**Time math, from the FT-r3 log (T4, Kaggle; measured) for ~800 rows x 2 epochs at effective batch 2**
| Item | Value | Basis |
|---|---|---|
| s/step | 40.1 s (297 steps in 11,906 s, mean ~6.3k tokens/row) | r3 metrics.json |
| training | 800 steps x 40.1 s = 32,080 s = 8.9 h | computed |
| eval | 663 s per full 116-row pass = 5.7 s/row; 40-row subset = 228 s; eval at start + every 100 steps = 9 passes = 0.57 h; full valid once at the end ~0.18 h | r3 log, computed |
| setup | install + model load + tokenise + llama.cpp build ~0.1 h | r3 log 230 s, build is an estimate |
| export | q4_k_m ~880 s in r3; with q8_0 ~0.3 h per exported checkpoint (estimate) | r3 log |
| **T4 total** | **about 10 h** for last-checkpoint export | computed |
| L4 (24 GB, bf16) | about 1.5-2x faster than T4: **~5-7 h** | estimate, not measured |
| A100 (40 GB) | about 4-6x faster than T4: **~2-2.5 h** | estimate, not measured |

Longer rows cost more than linearly: scale the step time by (mean_tokens / 6.3k)^1.15 (estimate). A full T4 run does not fit one free Colab session; use legs (`STOP_AT_STEP`, for example 130, then 450, then the end). Compute-unit cost per hour differs per GPU: check the units meter in the Colab UI before choosing.

## a. Settings

In [ ]:
RUN_NAME = 'r4'                      # drives OUT_DIR, adapter dirs, GGUF names, SHA256SUMS, metrics.json
DATA_NAME = 'ft_v7'
EXPECT_PROMPT_VERSION = 'v3'         # manifest and every row that carries the field must say this
EXPECT_SYSTEM_SHA256 = '52a9bda85eff7b0e4a9af1c97966e483e37d9c3d39fc3af2b35552887402b1aa'  # build_ft_v6.system_prompt('v3'); None skips (smoke runs on older data)
DRIVE_DIR = '/content/drive/MyDrive/buffettrag'
BUNDLE = DRIVE_DIR + f'/colab-bundle-7b-{DATA_NAME}.zip'
BUNDLE_DIR = '/content/bundle'
OUT_DIR = DRIVE_DIR + f'/run-7b-{RUN_NAME}'
CKPT_DIR = OUT_DIR + '/checkpoints'
ADAPTER_DIR = OUT_DIR + '/adapters'  # adapter-only copy of every saved step (small), kept forever
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
BASE_REVISION = None                 # HF commit hash of BASE_MODEL; fill from the first run (pins_resolved.json)
PINS = {'unsloth': '2026.10.6', 'transformers': '5.16.1', 'trl': '1.15.0',   # FT-r3 log: torch 2.11.0+cu128, transformers 5.16.1, trl 1.15.0, unsloth 2026.10.6
        'unsloth_zoo': None, 'peft': None, 'bitsandbytes': None}              # not in the r3 log: fill from pins_resolved.json
LLAMA_CPP_COMMIT = '46ca246de9bb1c35269722a6240d37d9dfd79cad'  # the llama.cpp the Mac serves with (models/llama.cpp-src); converter and quantizer are built from it
LLAMA_CPP_DIR = '/content/llama.cpp'
REQUIRE_ALL_PINS = True              # True: refuse to train while a pin above is None
MAX_SEQ_OVERRIDE = None              # None: real maximum token length of train+valid rounded up to 512
MAX_SEQ_CAP = 12288                  # above this the notebook stops instead of dropping rows
# LoRA / optimiser: FT-r1/FT-r3's: r=16, alpha=32, lr=1e-4, all 7 projections, batch 1 x grad-accum 2 = effective batch 2
EPOCHS = 2
LR = 1e-4
RANK = 16
ALPHA = 32
BATCH = 1
GRAD_ACCUM = 2
SEED = 13
SAVE_STEPS = 50                      # full checkpoint (resume point) + adapter copy every 50 steps (~33 min on a T4)
SAVE_TOTAL_LIMIT = 4                 # newest full checkpoints kept for resume (one may be half-written); adapters/step-N are never pruned
EVAL_STEPS = 100                     # eval at start, every EVAL_STEPS, at STOP_AT_STEP, on the fixed subset below
EVAL_ROWS = 40                       # seeded subset of valid (~4 min per pass at batch 1); the full valid runs once at the end
STOP_AT_STEP = None                  # int: end this leg cleanly at that optimizer step (checkpoint + eval + slice), no export
EXPORT_STEPS = []                    # extra checkpoint steps to export besides the last one (for example the best by the dev slice)
EXPORT_QUANTS = ['q4_k_m', 'q8_0']
ALLOW_EXISTING_EXPORT = False        # True only to finish an interrupted export
SLICE_FILE = DRIVE_DIR + '/slice_prompts.jsonl'   # rows: id, messages (system + user); optional
SLICE_STEPS = None                   # None: the step this run stops at (STOP_AT_STEP or the last); or a list of saved steps
SLICE_MAX_NEW_TOKENS = 900           # served limit; greedy (temperature 0)

## b. Helper functions (pure Python, no GPU)

In [ ]:
import glob, hashlib, json, math, os, random, re, shutil, struct, time, zipfile

RESUME_FILES = ('adapter_model.safetensors', 'adapter_config.json', 'trainer_state.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth')
ADAPTER_KEEP = ('adapter_model.safetensors', 'adapter_config.json', 'trainer_state.json')

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()

def parse_sums(path):
    out = {}
    for line in open(path):
        if line.strip():
            want, rel = line.strip().split(None, 1)
            out[rel.strip()] = want
    return out

def recorded_hashes(manifest):
    """sha256 values a manifest records for train/valid (same walk as scripts/ft/colab_bundle.py)."""
    found = {}
    def walk(node, path):
        if isinstance(node, dict):
            for key, val in node.items():
                walk(val, path + [str(key)])
        elif isinstance(node, str) and len(node) == 64 and all(c in '0123456789abcdef' for c in node):
            text = '/'.join(path).lower()
            if any(w in text for w in ('frozen', 'corpus', 'reuse', 'input')):
                return
            for split in ('train', 'valid'):
                if split in text and ('sha' in text or 'hash' in text):
                    found[split] = node
    walk(manifest, [])
    return found

def safetensors_complete(path):
    """The file is exactly header + the data the header promises (a half-copied file fails)."""
    try:
        size = os.path.getsize(path)
        with open(path, 'rb') as f:
            n = struct.unpack('<Q', f.read(8))[0]
            if n <= 0 or 8 + n > size:
                return False
            header = json.loads(f.read(n))
        end = max((v['data_offsets'][1] for k, v in header.items() if k != '__metadata__'), default=0)
        return end > 0 and size == 8 + n + end
    except Exception:
        return False

def checkpoint_problem(d):
    """None if the checkpoint dir is complete and loadable, else the reason."""
    for name in RESUME_FILES:
        p = os.path.join(d, name)
        if not os.path.isfile(p) or os.path.getsize(p) == 0:
            return f'missing or empty {name}'
    if not safetensors_complete(os.path.join(d, 'adapter_model.safetensors')):
        return 'adapter_model.safetensors incomplete'
    for name in ('optimizer.pt', 'scheduler.pt', 'rng_state.pth'):
        if not zipfile.is_zipfile(os.path.join(d, name)):      # torch.save writes a zip; a cut-off write has no central directory
            return f'{name} is not a complete torch file'
    try:
        state = json.load(open(os.path.join(d, 'trainer_state.json')))
        step = int(os.path.basename(d).split('-')[-1])
        if state.get('global_step') != step or not isinstance(state.get('log_history'), list):
            return 'trainer_state.json does not match the directory step'
    except Exception as e:
        return f'trainer_state.json unreadable: {e}'
    return None

def newest_complete_checkpoint(ckpt_dir):
    dirs = [d for d in glob.glob(ckpt_dir + '/checkpoint-*') if re.fullmatch(r'checkpoint-\d+', os.path.basename(d))]
    for d in sorted(dirs, key=lambda d: int(d.rsplit('-', 1)[1]), reverse=True):       # numeric: checkpoint-100 is newer than checkpoint-50
        why = checkpoint_problem(d)
        if why is None:
            return d
        print('skipping incomplete checkpoint', d, '->', why)
    return None

def copy_adapter(src, dst):
    os.makedirs(dst, exist_ok=True)
    for name in ADAPTER_KEEP:
        shutil.copy(os.path.join(src, name), os.path.join(dst, name))
    assert safetensors_complete(os.path.join(dst, 'adapter_model.safetensors')), 'adapter copy incomplete: ' + dst

def read_json(p):
    return json.load(open(p))

def update_json(p, **fields):
    cur = read_json(p) if os.path.isfile(p) else {}
    cur.update(fields)
    tmp = p + '.part'
    json.dump(cur, open(tmp, 'w'), indent=2)
    os.replace(tmp, p)
    return cur

def run_guard(out_dir, run_name, allow_existing_export=False):
    """Refuse a directory that belongs to another run or already holds a GGUF (C1, TRAIN-05, EXP-03). Returns run_meta (or {})."""
    assert os.path.basename(out_dir.rstrip('/')) == f'run-7b-{run_name}', f'{out_dir} is not run-7b-{run_name}'
    meta = read_json(out_dir + '/run_meta.json') if os.path.isfile(out_dir + '/run_meta.json') else {}
    for found in (meta.get('run_name'), read_json(out_dir + '/metrics.json').get('run_name') if os.path.isfile(out_dir + '/metrics.json') else None):
        if found is not None and found != run_name:
            raise SystemExit(f'{out_dir} holds a run named {found!r}, this notebook is {run_name!r}: use a fresh RUN_NAME/OUT_DIR')
    ggufs = glob.glob(out_dir + '/*.gguf')
    foreign = [g for g in ggufs if f'-{run_name}-' not in os.path.basename(g)]
    if foreign:
        raise SystemExit(f'GGUF(s) of another run in {out_dir}: {foreign}')
    if ggufs and not allow_existing_export:
        raise SystemExit(f'{len(ggufs)} GGUF(s) already exist in {out_dir}; set ALLOW_EXISTING_EXPORT = True only to finish an interrupted export')
    return meta

def check_identity(meta, **now):
    """A resume must see the same data, seed and run as the run that wrote the checkpoint."""
    for k, v in now.items():
        if k in meta and meta[k] != v:
            raise SystemExit(f'run_meta {k} differs from this session ({meta[k]!r} != {v!r}); refusing to resume on different data/settings')

def pick_eval_subset(kinds, n, seed):
    """Fixed, seeded, kind-balanced indices of n valid rows (round robin over kinds; all rows if n >= len)."""
    idx = list(range(len(kinds)))
    if n >= len(idx):
        return idx
    rng = random.Random(seed)
    groups = {}
    for i in idx:
        groups.setdefault(kinds[i], []).append(i)
    for g in groups.values():
        rng.shuffle(g)
    order = sorted(groups)
    picked = []
    while len(picked) < n:
        for k in order:
            if groups[k] and len(picked) < n:
                picked.append(groups[k].pop())
    return sorted(picked)

def histogram_text(lengths, step=1024, width=40):
    bins = {}
    for n in lengths:
        bins[n // step] = bins.get(n // step, 0) + 1
    peak = max(bins.values())
    return '\n'.join(f'{b * step:>6}-{(b + 1) * step - 1:<6} {c:>4} ' + '#' * max(1, round(width * c / peak)) for b, c in sorted(bins.items()))

def estimate_peak_gb(max_seq, ref_seq=9728, ref_peak=7.7, static=5.5):
    """Linear extrapolation of FT-r3's measured peak (7.7 GB at MAX_SEQ 9728, T4); static=4-bit weights + LoRA state (estimate)."""
    return round(static + (ref_peak - static) * max_seq / ref_seq, 2)

def check_memory_fits(max_seq, total_gb, share=0.85):
    est = estimate_peak_gb(max_seq)
    if est > share * total_gb:
        raise SystemExit(f'estimated peak {est} GB for MAX_SEQ={max_seq} does not fit {total_gb:.1f} GB GPU memory (limit {share:.0%})')
    return est

## c. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > GPU, then re-run.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU:', GPU_NAME, round(GPU_TOTAL_GB, 1), 'GB')

## d. Mount Drive, run guard, unzip bundle, verify SHA256SUMS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
RUN_META = run_guard(OUT_DIR, RUN_NAME, ALLOW_EXISTING_EXPORT)   # SystemExit on a foreign run or an existing GGUF
assert os.path.isfile(BUNDLE), 'bundle not found: ' + BUNDLE
shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall(BUNDLE_DIR)
bad = 0
for rel, want in parse_sums(BUNDLE_DIR + '/SHA256SUMS').items():
    ok = sha256_file(BUNDLE_DIR + '/' + rel) == want
    bad += not ok
    print('OK ' if ok else 'BAD', rel)
if bad:
    raise SystemExit(f'{bad} checksum mismatch(es); stopping.')
print('run:', RUN_NAME, '| out:', OUT_DIR, '| earlier run_meta:', {k: RUN_META.get(k) for k in ('status', 'train_sha256')} if RUN_META else 'none')

## e. Install pinned libraries, verify versions, build the pinned llama.cpp

In [ ]:
import importlib.metadata as md_, subprocess, sys
missing = [k for k, v in PINS.items() if not v]
if (missing or not BASE_REVISION) and REQUIRE_ALL_PINS:
    raise SystemExit(f'unfilled pins: {missing + ([] if BASE_REVISION else ["BASE_REVISION"])}. Fill section a, or set REQUIRE_ALL_PINS = False for a smoke run (writes pins_resolved.json).')
specs = [f'{k}=={v}' if v else k for k, v in PINS.items()]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *specs], check=True)
VERSIONS = {k: md_.version(k) for k in ('torch', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets')}
for k, v in PINS.items():
    assert not v or VERSIONS[k] == v, f'{k} is {VERSIONS[k]}, pinned {v}'
print(VERSIONS)
FREEZE = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
open(OUT_DIR + '/pip_freeze.txt', 'w').write(FREEZE)

def git(*a, cwd=LLAMA_CPP_DIR):
    return subprocess.run(['git', *a], cwd=cwd, capture_output=True, text=True, check=True).stdout.strip()

if not os.path.isdir(LLAMA_CPP_DIR + '/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ggml-org/llama.cpp', LLAMA_CPP_DIR], check=True)
git('checkout', '-q', LLAMA_CPP_COMMIT)
assert git('rev-parse', 'HEAD') == LLAMA_CPP_COMMIT
if not os.path.isfile(LLAMA_CPP_DIR + '/llama-quantize'):
    subprocess.run(['cmake', '-B', 'build', '-DGGML_CUDA=OFF', '-DLLAMA_CURL=OFF'], cwd=LLAMA_CPP_DIR, check=True, capture_output=True)
    subprocess.run(['cmake', '--build', 'build', '--target', 'llama-quantize', '-j', '4'], cwd=LLAMA_CPP_DIR, check=True, capture_output=True)
    shutil.copy(LLAMA_CPP_DIR + '/build/bin/llama-quantize', LLAMA_CPP_DIR + '/llama-quantize')
print('llama.cpp', git('rev-parse', 'HEAD'))

## f. Load base model + LoRA (max length = the cap; the real MAX_SEQ is set from the data in the next step)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ_CAP, load_in_4bit=True, revision=BASE_REVISION)
from huggingface_hub import model_info
BASE_SHA = model_info(BASE_MODEL, revision=BASE_REVISION).sha     # the exact base revision this run used
assert not BASE_REVISION or BASE_SHA == BASE_REVISION, (BASE_SHA, BASE_REVISION)
print('base model', BASE_MODEL, BASE_SHA)
update_json(OUT_DIR + '/pins_resolved.json', versions=VERSIONS, base_model=BASE_MODEL, base_revision=BASE_SHA, llama_cpp_commit=LLAMA_CPP_COMMIT)
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=ALPHA, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)

## g. Data: identity (sha256, prompt v3), serving chat template, MAX_SEQ = real maximum, no dropped rows

In [ ]:
import json, math
from datasets import Dataset

DATA_DIR = f'{BUNDLE_DIR}/{DATA_NAME}'
MANIFEST = json.load(open(f'{DATA_DIR}/manifest.json'))
assert MANIFEST.get('prompt_version') == EXPECT_PROMPT_VERSION, MANIFEST.get('prompt_version')
DATA_SHA = {s: sha256_file(f'{DATA_DIR}/{s}.jsonl') for s in ('train', 'valid')}
SUMS = parse_sums(BUNDLE_DIR + '/SHA256SUMS')
for s, got in DATA_SHA.items():
    assert SUMS.get(f'{DATA_NAME}/{s}.jsonl') == got, f'{s}.jsonl is not the file SHA256SUMS lists'
    want = recorded_hashes(MANIFEST).get(s)
    assert want is None or want == got, f'{s}.jsonl sha256 differs from the manifest ({want} != {got})'
print('data sha256:', DATA_SHA, '| manifest prompt_version', MANIFEST['prompt_version'], '| passages_mix', MANIFEST.get('passages_mix'), '| k_histogram', MANIFEST.get('k_histogram'))

def load(path):
    return [json.loads(l) for l in open(path) if l.strip()]

def encode(rows):
    # ChatML via the tokenizer's own template, exactly what llama.cpp applies at serving: system, user, then the assistant answer
    out = []
    for r in rows:
        msgs = r['messages']
        assert [m['role'] for m in msgs] == ['system', 'user', 'assistant'], [m['role'] for m in msgs]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        out.append((text, len(tokenizer(text, add_special_tokens=False)['input_ids'])))
    return out

train_full = load(f'{DATA_DIR}/train.jsonl')
valid_full = load(f'{DATA_DIR}/valid.jsonl')
train_rows = [r['messages'] for r in train_full]
valid_rows = [r['messages'] for r in valid_full]
with_field = [r for r in train_full + valid_full if 'prompt_version' in r]
assert all(r['prompt_version'] == EXPECT_PROMPT_VERSION for r in with_field), 'a row carries another prompt_version'
print(f'prompt_version == {EXPECT_PROMPT_VERSION!r} on {len(with_field)}/{len(train_full) + len(valid_full)} rows that carry the field')
systems = {m[0]['content'] for m in train_rows + valid_rows}
assert len(systems) == 1, f'{len(systems)} different system prompts'
SYSTEM = next(iter(systems))
SYSTEM_SHA = hashlib.sha256(SYSTEM.encode()).hexdigest()
assert EXPECT_SYSTEM_SHA256 is None or SYSTEM_SHA == EXPECT_SYSTEM_SHA256, f'system prompt sha256 {SYSTEM_SHA} is not the {EXPECT_PROMPT_VERSION} prompt'
assert EXPECT_PROMPT_VERSION != 'v3' or 'Partnership' not in SYSTEM, 'v3 system prompt must not mention the Partnership letters'
print('system prompt (one for all rows):', SYSTEM[:90].replace(chr(10), ' '), '...', SYSTEM_SHA[:12])
assert all(m[1]['content'].lstrip().startswith(('BEGIN UNTRUSTED PASSAGES', 'BEGIN RECENT CONVERSATION')) for m in train_rows + valid_rows)

train_enc, valid_enc = encode(train_full), encode(valid_full)
lengths = sorted(n for _, n in train_enc + valid_enc)
P50, P95, P99 = (lengths[min(len(lengths) - 1, math.ceil(q / 100 * len(lengths)) - 1)] for q in (50, 95, 99))  # nearest rank
MAX_LEN = lengths[-1]
MAX_SEQ = MAX_SEQ_OVERRIDE or math.ceil(MAX_LEN / 512) * 512      # the real maximum (train and valid), not p99
assert MAX_SEQ >= MAX_LEN, f'MAX_SEQ_OVERRIDE={MAX_SEQ_OVERRIDE} is shorter than the longest row ({MAX_LEN}): rows would be dropped'
assert MAX_SEQ <= MAX_SEQ_CAP, f'longest row is {MAX_LEN} tokens, above MAX_SEQ_CAP={MAX_SEQ_CAP}; shorten the rows or raise the cap after a memory check'
print(f'tokens prompt+answer: p50={P50} p95={P95} p99={P99} max={MAX_LEN} -> MAX_SEQ={MAX_SEQ} (cap {MAX_SEQ_CAP})')
print(histogram_text(lengths))

def keep(enc, name):
    kept = [{'text': t} for t, n in enc if n <= MAX_SEQ]
    dropped = len(enc) - len(kept)
    print(f'{name}: rows={len(enc)} kept={len(kept)} dropped(>{MAX_SEQ})={dropped}')
    assert dropped == 0, f'{name}: {dropped} rows are longer than MAX_SEQ'
    return Dataset.from_list(kept), dropped

train_ds, TRAIN_DROPPED = keep(train_enc, 'train')
valid_ds, VALID_DROPPED = keep(valid_enc, 'valid')
# fixed seeded subset of valid for the step evals (kind-balanced when rows carry a kind); full valid runs once at the end
EVAL_IDX = pick_eval_subset([r.get('kind') for r in valid_full], EVAL_ROWS, SEED)
eval_sub_ds = Dataset.from_list([{'text': valid_enc[i][0]} for i in EVAL_IDX])
EVAL_IDS = [valid_full[i].get('id', i) for i in EVAL_IDX]
print(f'step-eval subset: {len(EVAL_IDX)} of {len(valid_full)} valid rows (seed {SEED})')

## h. Memory check before the long run (estimate, then one forward/backward of the longest row)

In [ ]:
EST_PEAK_GB = check_memory_fits(MAX_SEQ, GPU_TOTAL_GB)
print(f'estimated peak {EST_PEAK_GB} GB for MAX_SEQ={MAX_SEQ} on {GPU_TOTAL_GB:.1f} GB (FT-r3: 7.7 GB at 9728)')
longest = max(train_enc + valid_enc, key=lambda t: t[1])
ids = tokenizer(longest[0], add_special_tokens=False, return_tensors='pt')['input_ids'].cuda()
model.train()
torch.cuda.reset_peak_memory_stats()
with torch.autocast('cuda', dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
    probe_loss = model(input_ids=ids, labels=ids).loss
probe_loss.backward()
model.zero_grad(set_to_none=True)
PROBE_PEAK_GB = round(torch.cuda.max_memory_allocated() / 2**30, 2)
del ids, probe_loss
torch.cuda.empty_cache()
print(f'probe: longest row ({longest[1]} tokens) forward+backward peak {PROBE_PEAK_GB} GB of {GPU_TOTAL_GB:.1f}')
assert PROBE_PEAK_GB <= 0.92 * GPU_TOTAL_GB, 'the longest row does not fit with headroom'

## i. Train (auto-resumes from the newest COMPLETE checkpoint on Drive)

In [ ]:
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.chat_templates import train_on_responses_only

STEPS_PER_EPOCH = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM))
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
EPOCH_END_STEPS = {STEPS_PER_EPOCH * e for e in range(1, EPOCHS + 1)}        # always saved, so adapters/step-N exists for each epoch end and the last step
assert STOP_AT_STEP is None or 0 < STOP_AT_STEP <= TOTAL_STEPS, (STOP_AT_STEP, TOTAL_STEPS)
os.makedirs(ADAPTER_DIR, exist_ok=True)
resume_path = newest_complete_checkpoint(CKPT_DIR)
check_identity(RUN_META, train_sha256=DATA_SHA['train'], valid_sha256=DATA_SHA['valid'], seed=SEED, data_name=DATA_NAME) if resume_path else None
RUN_META = update_json(OUT_DIR + '/run_meta.json', run_name=RUN_NAME, status='training', data_name=DATA_NAME, train_sha256=DATA_SHA['train'], valid_sha256=DATA_SHA['valid'],
                       seed=SEED, steps_per_epoch=STEPS_PER_EPOCH, total_steps=TOTAL_STEPS, eval_idx=EVAL_IDX)

class R4Callback(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step in EPOCH_END_STEPS:
            control.should_save = True
        if STOP_AT_STEP and state.global_step >= STOP_AT_STEP:
            control.should_save = control.should_evaluate = control.should_training_stop = True
        return control
    def on_save(self, args, state, control, **kw):
        src = f'{args.output_dir}/checkpoint-{state.global_step}'
        copy_adapter(src, f'{ADAPTER_DIR}/step-{state.global_step}')          # adapter-only copy, never pruned
        print('saved checkpoint + adapters/step-%d' % state.global_step)

args = SFTConfig(
    output_dir=CKPT_DIR, dataset_text_field='text', max_seq_length=MAX_SEQ, packing=False,
    num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, per_device_eval_batch_size=1, logging_steps=5,
    eval_strategy='steps', eval_steps=EVAL_STEPS, eval_on_start=resume_path is None,
    save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=SAVE_TOTAL_LIMIT,
    fp16=not is_bf16_supported(), bf16=is_bf16_supported(), seed=SEED, data_seed=SEED,
    optim='adamw_8bit', lr_scheduler_type='linear', warmup_steps=5, weight_decay=0.01, report_to='none')
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=eval_sub_ds, args=args, callbacks=[R4Callback()])
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
# loss on the assistant answer only: what is left unmasked must be exactly the answer (+ <|im_end|>)
for i in range(3):
    ex = trainer.train_dataset[i]
    live = tokenizer.decode([t for t in ex['labels'] if t != -100]).strip()
    answer = train_rows[i][2]['content'].strip()
    assert live == answer + '<|im_end|>' or live == answer, (live[:120], answer[:120])
print('assistant-only loss check OK on 3 rows')
print('resume from checkpoint:', resume_path, '| steps/epoch', STEPS_PER_EPOCH, '| total steps', TOTAL_STEPS, '| stop at', STOP_AT_STEP)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
result = trainer.train(resume_from_checkpoint=resume_path)
SECONDS = round(time.time() - t0, 1)
PEAK_GPU_GB = round(torch.cuda.max_memory_allocated() / 2**30, 2)
FINAL_STEP = trainer.state.global_step
LEG_STOPPED = FINAL_STEP < TOTAL_STEPS
print(result)
print('step', FINAL_STEP, 'of', TOTAL_STEPS, '| leg stopped early:', LEG_STOPPED, '| peak GPU memory allocated (GB):', PEAK_GPU_GB, 'of', round(GPU_TOTAL_GB, 1))

## j. Final adapter, full-valid eval, metrics (a stopped leg writes only its leg file)

In [ ]:
hist = trainer.state.log_history
evals = [{'step': h['step'], 'epoch': round(h['epoch'], 3), 'eval_loss': h['eval_loss']} for h in hist if 'eval_loss' in h]
train_losses = [h['loss'] for h in hist if 'loss' in h]
if LEG_STOPPED:
    update_json(OUT_DIR + f'/leg_{FINAL_STEP}.json', run_name=RUN_NAME, step=FINAL_STEP, seconds=SECONDS, peak_gpu_gb=PEAK_GPU_GB, eval_history_subset=evals,
                last_logged_train_loss=train_losses[-1] if train_losses else None, adapter=f'adapters/step-{FINAL_STEP}')
    update_json(OUT_DIR + '/run_meta.json', status=f'stopped_at_{FINAL_STEP}')
    print('leg done at step', FINAL_STEP, '; eval history', evals)
else:
    model.save_pretrained(OUT_DIR + '/adapter')
    tokenizer.save_pretrained(OUT_DIR + '/adapter')
    assert os.path.isfile(f'{ADAPTER_DIR}/step-{FINAL_STEP}/adapter_model.safetensors'), 'last-step adapter copy missing'
    # full valid once: the trainer masks labels only on the datasets it holds, so swap the full set in and mask it the same way
    trainer.eval_dataset = valid_ds
    trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
    assert any(t == -100 for t in trainer.eval_dataset[0]['labels']), 'full valid labels are not masked'
    full = trainer.evaluate(metric_key_prefix='full_valid')
    FULL_VALID_LOSS = full['full_valid_loss']
    metrics = {
        'run_name': RUN_NAME, 'gpu': GPU_NAME, 'versions': VERSIONS, 'pins': PINS, 'base_model': BASE_MODEL, 'base_revision': BASE_SHA, 'llama_cpp_commit': LLAMA_CPP_COMMIT,
        'seed': SEED, 'data_name': DATA_NAME, 'data_sha256': DATA_SHA, 'prompt_version': MANIFEST.get('prompt_version'), 'system_prompt_sha256': SYSTEM_SHA,
        'max_seq': MAX_SEQ, 'max_seq_cap': MAX_SEQ_CAP, 'token_p50_p95_p99_max': [P50, P95, P99, MAX_LEN], 'peak_gpu_gb': PEAK_GPU_GB, 'est_peak_gb': EST_PEAK_GB, 'probe_peak_gb': PROBE_PEAK_GB,
        'lora': {'rank': RANK, 'alpha': ALPHA, 'lr': LR, 'batch': BATCH, 'grad_accum': GRAD_ACCUM}, 'manifest_k_histogram': MANIFEST.get('k_histogram'),
        'epochs_planned': EPOCHS, 'epochs_completed': round(trainer.state.epoch or 0, 3), 'steps': FINAL_STEP, 'total_steps': TOTAL_STEPS, 'train_rows': len(train_ds),
        'dropped_rows': {'train': TRAIN_DROPPED, 'valid': VALID_DROPPED},
        'final_train_loss': result.training_loss if result.training_loss is not None else (train_losses[-1] if train_losses else None),
        'last_logged_train_loss': train_losses[-1] if train_losses else None,
        'eval_rows_subset': len(EVAL_IDX), 'eval_subset_ids': EVAL_IDS, 'eval_loss_history_subset': evals, 'full_valid_eval_loss': FULL_VALID_LOSS, 'full_valid_rows': len(valid_ds),
        'save_steps': SAVE_STEPS, 'eval_steps': EVAL_STEPS,
        'seconds': SECONDS, 'seconds_note': 'this session only (a resumed run counts the resumed part)', 'exports': {},
    }
    json.dump(metrics, open(OUT_DIR + '/metrics.json', 'w'), indent=2)
    update_json(OUT_DIR + '/run_meta.json', status='trained')
    print(json.dumps(metrics, indent=2))

## k. Dev-slice generation (optional): greedy answers per saved checkpoint, so the Mac can score them without a GGUF

In [ ]:
import gc
globals().pop('trainer', None); globals().pop('model', None)
gc.collect(); torch.cuda.empty_cache()
SLICE_RESULTS = {}
if not os.path.isfile(SLICE_FILE):
    print('no slice_prompts.jsonl on Drive, skipping')
else:
    slice_rows = [json.loads(l) for l in open(SLICE_FILE) if l.strip()]
    steps = SLICE_STEPS if SLICE_STEPS is not None else [FINAL_STEP]
    for step in steps:
        adir = f'{ADAPTER_DIR}/step-{step}'
        dst = f'{OUT_DIR}/slice_answers_{step}.jsonl'
        assert os.path.isdir(adir), f'no saved adapter for step {step}: {adir}'
        if os.path.isfile(dst):
            print('exists, skipping:', dst)
            continue
        sm, st = FastLanguageModel.from_pretrained(adir, max_seq_length=MAX_SEQ, load_in_4bit=True, revision=BASE_REVISION)
        FastLanguageModel.for_inference(sm)
        out = []
        for r in slice_rows:
            msgs = [m for m in r['messages'] if m['role'] != 'assistant']
            ids = st.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').cuda()
            assert ids.shape[1] <= MAX_SEQ, f"slice prompt {r['id']} is {ids.shape[1]} tokens, above MAX_SEQ"
            with torch.no_grad():
                gen = sm.generate(input_ids=ids, max_new_tokens=SLICE_MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None)
            out.append({'id': r['id'], 'step': step, 'prompt_tokens': int(ids.shape[1]), 'answer': st.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip()})
        with open(dst + '.part', 'w') as f:
            f.write(''.join(json.dumps(o) + '\n' for o in out))
        os.replace(dst + '.part', dst)
        SLICE_RESULTS[step] = dst
        print('wrote', dst, len(out), 'answers')
        del sm, st
        gc.collect(); torch.cuda.empty_cache()

## l. Export GGUF (q4_k_m and q8_0) for the last checkpoint and every step in EXPORT_STEPS, with SHA256SUMS

In [ ]:
if LEG_STOPPED:
    print(f'leg stopped at step {FINAL_STEP}: no export (re-run with STOP_AT_STEP = None to finish)')
else:
    EXPORT_LIST = sorted({FINAL_STEP, *EXPORT_STEPS})
    for s in EXPORT_LIST:
        assert safetensors_complete(f'{ADAPTER_DIR}/step-{s}/adapter_model.safetensors'), f'no complete saved adapter for step {s}'

    def gguf_name(step, quant):
        return f'{OUT_DIR}/buffett-qwen2.5-7b-ft-{RUN_NAME}-s{step}-{quant}.gguf'

    def write_sums():
        with open(OUT_DIR + '/SHA256SUMS', 'w') as f:
            for p in sorted(glob.glob(OUT_DIR + '/*.gguf')):
                f.write(f'{sha256_file(p)}  {os.path.basename(p)}\n')

    def export_ggufs(m, tok, work, step):
        # unsloth merges the LoRA into a temp dir relative to the cwd, so work in /tmp and clean up after copying; llama.cpp is the pinned build
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work)
        os.symlink(LLAMA_CPP_DIR, work + '/llama.cpp')
        cwd = os.getcwd()
        os.chdir(work)
        try:
            m.save_pretrained_gguf(work + '/out', tok, quantization_method=EXPORT_QUANTS)
            assert git('rev-parse', 'HEAD', cwd=work + '/llama.cpp') == LLAMA_CPP_COMMIT, 'export did not use the pinned llama.cpp'
            for q in EXPORT_QUANTS:
                cands = [p for p in glob.glob(work + '/**/*.gguf', recursive=True) if q in os.path.basename(p).lower()]
                assert len(cands) == 1, f'expected one {q} gguf, found {cands}'
                shutil.copy(cands[0], gguf_name(step, q) + '.part')
                os.replace(gguf_name(step, q) + '.part', gguf_name(step, q))
        finally:
            os.chdir(cwd)
            shutil.rmtree(work, ignore_errors=True)

    metrics = read_json(OUT_DIR + '/metrics.json')
    for s in EXPORT_LIST:
        if all(os.path.exists(gguf_name(s, q)) for q in EXPORT_QUANTS):
            print('exists, skipping step', s)
        else:
            m, tok = FastLanguageModel.from_pretrained(f'{ADAPTER_DIR}/step-{s}', max_seq_length=MAX_SEQ, load_in_4bit=True, revision=BASE_REVISION)
            try:
                export_ggufs(m, tok, '/tmp/gguf_ft', s)
            finally:
                del m, tok
                gc.collect(); torch.cuda.empty_cache()
        for q in EXPORT_QUANTS:
            p = gguf_name(s, q)
            metrics['exports'][os.path.basename(p)] = {'step': s, 'quant': q, 'bytes': os.path.getsize(p), 'sha256': sha256_file(p)}
            print(p, os.path.getsize(p), 'bytes\nsha256', metrics['exports'][os.path.basename(p)]['sha256'])
    write_sums()
    metrics['llama_cpp_commit_exported_with'] = LLAMA_CPP_COMMIT
    json.dump(metrics, open(OUT_DIR + '/metrics.json', 'w'), indent=2)
    update_json(OUT_DIR + '/run_meta.json', status='finished', exported_steps=EXPORT_LIST)
    print(open(OUT_DIR + '/SHA256SUMS').read())

## m. Done

In [ ]:
print('ALL_DONE' if not LEG_STOPPED else f'LEG_DONE step={FINAL_STEP}')
for root, _, files in os.walk(OUT_DIR):
    if '/checkpoints' in root:
        continue
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p):>14,}  {p}')